# Four-state check, 2025: Pennsylvania, Wisconsin, Iowa, Nebraska

Runs `agents/states/pipeline.py` in the background: download the NASS field outlines (2018-2025), sample about 300 corn fields per state, extract Sentinel-2, classify with `eu_logistic_v1` and `eu_barc_logistic_v1`, compare with the 2025 USDA survey (silage share and acres) and 2025 harvest progress. No Nemotron calls. Uses all CPUs. About 1.5-2.5 hours.

In [ ]:
# 1. Start in the background (safe to close this tab)
import os, subprocess, sys
from pathlib import Path
from fieldvision_agents.settings import REPO; os.chdir(REPO); Path('data/states').mkdir(parents=True, exist_ok=True)
log = open('data/states/pipeline.log', 'a')
proc = subprocess.Popen([sys.executable, '-u', '-m', 'fieldvision_agents.states.pipeline', 'all'], cwd=REPO,
                        stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print('started, pid', proc.pid)

In [ ]:
# 2. Progress (rerun any time)
print(open('data/states/pipeline.log').read()[-3000:])

In [ ]:
# 3. Results, once the log says done
import json, pandas as pd
c = json.load(open('data/states/comparison.json'))
print(json.dumps(c['summary'], indent=1))
pd.DataFrame(c['shares'])

In [ ]:
# 4. Stop everything this pipeline started (only if needed)
import os, signal
pids = [int(p) for p in os.listdir('/proc') if p.isdigit() and b'fieldvision_agents.states.pipeline' in open(f'/proc/{p}/cmdline', 'rb').read()]
for pid in pids:
    try:
        os.killpg(os.getpgid(pid), signal.SIGTERM)
    except ProcessLookupError:
        pass
print('stopped', pids)